# 12B — วินิจฉัยสภาพบรรยากาศรายวัน

## กรณีศึกษา: พายุโซนร้อนเตี้ยนหมู่ 24 กันยายน 2564

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**บทเสริม · ใช้เวลาประมาณ 3 ชั่วโมง · เรียนหลังบทที่ 12A**

---

## โจทย์

วันที่ 24 กันยายน 2564 บรรยากาศเหนือภูมิภาคนี้เป็นอย่างไร

เราจะไม่สนใจว่าพายุเคลื่อนที่ไปทางไหน แต่จะ **ผ่าบรรยากาศของวันนั้นออกดูทีละชั้น**
เหมือนหมออ่านผลตรวจร่างกาย — ดูทีละตัวแปรแล้วประกอบเป็นภาพรวม

ERA5 ให้ตัวแปรดิบมาแค่ 6 ตัว (`t2m`, `msl`, `u`, `v`, `z`, `tp`)
แต่เราจะคำนวณต่อให้ได้ตัวแปรวินิจฉัยอีก 8 ตัวที่นักอุตุนิยมวิทยาใช้จริง

| ตัวแปรที่จะคำนวณเอง | บอกอะไร |
|---|---|
| wind speed / direction | ความแรงและทิศทางลม |
| relative vorticity | ความหมุนวนของอากาศ |
| divergence | อากาศไหลเข้าหรือออกจากกัน |
| deep layer shear | ลมเปลี่ยนตามความสูงมากแค่ไหน |
| thickness | ชั้นอากาศหนาหรือบาง |
| layer mean temperature | อุณหภูมิเฉลี่ยของชั้น |
| temperature advection | ลมพาอากาศอุ่นหรือเย็นเข้ามา |
| geostrophic wind | ลมตามทฤษฎีสมดุล เทียบกับลมจริง |

---

> **วิธีใช้บทนี้**
> เซลล์ในบทนี้มีแต่โค้ดกับหัวข้อ **ไม่มีคำอธิบายผลลัพธ์ให้**
> หน้าที่ของคุณคือรันแล้ว **สังเกตเอง ตีความเอง และจดบันทึก**
>
> ใต้ทุกหัวข้อจะมีช่อง ✍️ ให้เขียนสิ่งที่สังเกตได้
> ผู้สอนจะเฉลยการตีความหลังจากทุกคนลองคิดเองแล้ว

## 0. เตรียมข้อมูลและเครื่องมือ

In [ ]:
!pip install -q cartopy
print("ติดตั้ง cartopy เรียบร้อย")

In [ ]:
import os, urllib.request

BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

os.makedirs("data/cases", exist_ok=True)
if not os.path.exists("data/cases/dianmu2021.nc"):
    urllib.request.urlretrieve(BASE + "data/cases/dianmu2021.nc",
                               "data/cases/dianmu2021.nc")
if not os.path.exists("atmos_diag.py"):
    urllib.request.urlretrieve(BASE + "utils/atmos_diag.py", "atmos_diag.py")

import matplotlib, matplotlib.font_manager as fm
try:
    if not os.path.exists("Sarabun-Regular.ttf"):
        urllib.request.urlretrieve(
            "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf",
            "Sarabun-Regular.ttf")
    fm.fontManager.addfont("Sarabun-Regular.ttf")
    matplotlib.rc("font", family="Sarabun")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as exc:
    print("ตั้งฟอนต์ไทยไม่สำเร็จ:", exc)

print("พร้อมแล้ว")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import atmos_diag as ad

plt.rcParams["figure.dpi"] = 110


def add_basemap(ax, extent, ocean=True, land=True, gridlines=True):
    """เตรียมแผนที่ฐาน (ฟังก์ชันเดียวกับที่เขียนไว้ในบทที่ 12A)"""
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    try:
        for feat in (cfeature.COASTLINE, cfeature.BORDERS,
                     cfeature.OCEAN, cfeature.LAND):
            next(iter(feat.geometries()), None)
        if ocean:
            ax.add_feature(cfeature.OCEAN, facecolor="#eaf3fb", zorder=0)
        if land:
            ax.add_feature(cfeature.LAND, facecolor="#f7f5f0", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
    except Exception as exc:
        print("โหลดแผนที่ฐานไม่ได้ ->", exc)
    if gridlines:
        gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                          alpha=0.5, linestyle="--")
        gl.top_labels = gl.right_labels = False
    return ax


# ---- ค่าตั้งต้นของบทนี้ ----
ds = xr.open_dataset("data/cases/dianmu2021.nc")
if float(ds["latitude"][0]) > float(ds["latitude"][-1]):
    ds = ds.sortby("latitude")

TIME = "2021-09-24T12:00"          # เวลาหลักที่จะวิเคราะห์ (UTC)
DOMAIN = [95, 115, 5, 25]
UBON = (15.25, 104.87)             # จุดอ้างอิง จังหวัดอุบลราชธานี

day = ds.sel(time=TIME)

print("ตัวแปรในไฟล์ :", list(ds.data_vars))
print("จำนวนช่วงเวลา:", ds.sizes["time"])
print("ระดับความกดอากาศ:", ds["level"].values, "hPa")
print("เวลาที่วิเคราะห์:", TIME, "UTC  =", 
      (pd.Timestamp(TIME) + pd.Timedelta(hours=7)).strftime("%d/%m/%Y %H:%M"), "เวลาไทย")

## 1. ภาพรวมผิวพื้น — ความกดอากาศ อุณหภูมิ และฝน

In [ ]:
p = day["msl"] / 100          # Pa -> hPa
t = day["t2m"] - 273.15       # K -> C
r = day["tp"]                 # mm ต่อ 6 ชั่วโมง

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax in axes:
    add_basemap(ax, DOMAIN, gridlines=False)

c0 = axes[0].contourf(p["longitude"], p["latitude"], p.values,
                      levels=np.arange(996, 1017, 1), cmap="viridis_r",
                      extend="both", transform=ccrs.PlateCarree())
plt.colorbar(c0, ax=axes[0], shrink=0.75, label="hPa")
axes[0].set_title("ความกดอากาศระดับน้ำทะเล", fontsize=11, loc="left")

c1 = axes[1].contourf(t["longitude"], t["latitude"], t.values,
                      levels=np.arange(18, 33, 1), cmap="RdYlBu_r",
                      extend="both", transform=ccrs.PlateCarree())
plt.colorbar(c1, ax=axes[1], shrink=0.75, label="°C")
axes[1].set_title("อุณหภูมิที่ 2 เมตร", fontsize=11, loc="left")

c2 = axes[2].contourf(r["longitude"], r["latitude"], r.values,
                      levels=[1, 2, 5, 10, 15, 20, 25], cmap="GnBu",
                      extend="max", transform=ccrs.PlateCarree())
plt.colorbar(c2, ax=axes[2], shrink=0.75, label="mm / 6 ชม.")
axes[2].set_title("ฝนสะสม 6 ชั่วโมง", fontsize=11, loc="left")

fig.suptitle(f"ภาพรวมผิวพื้น {TIME} UTC", fontsize=13)
plt.show()

low = ad.extreme_location(p, "min")
rain_max = ad.extreme_location(r, "max")
print(f"ความกดอากาศต่ำสุด {low['value']:.1f} hPa ที่ {low['lat']:.1f}N {low['lon']:.1f}E")
print(f"ฝนสูงสุด         {rain_max['value']:.1f} mm ที่ {rain_max['lat']:.1f}N {rain_max['lon']:.1f}E")
print(f"อุณหภูมิ {float(t.min()):.1f} ถึง {float(t.max()):.1f} °C "
      f"| ที่อุบลราชธานี {ad.value_at(t, *UBON):.1f} °C")

### ✍️ บันทึกการสังเกต

สังเกตอะไรได้บ้างจากสามแผนที่นี้ ตำแหน่งความกดอากาศต่ำสุดกับฝนหนักสุดตรงกันไหม

*(เขียนคำตอบที่นี่)*

## 2. โครงสร้างแนวดิ่ง — ความสูง geopotential ทุกระดับ

In [ ]:
levels_hpa = [1000, 925, 850, 700, 500, 200]

fig, axes = plt.subplots(2, 3, figsize=(16, 9),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax, lev in zip(axes.flat, levels_hpa):
    gh = ad.geopotential_height(day["z"].sel(level=lev))
    add_basemap(ax, DOMAIN, gridlines=False)

    cf = ax.contourf(gh["longitude"], gh["latitude"], gh.values,
                     levels=12, cmap="Spectral_r",
                     transform=ccrs.PlateCarree())
    cs = ax.contour(gh["longitude"], gh["latitude"], gh.values,
                    levels=12, colors="black", linewidths=0.5,
                    transform=ccrs.PlateCarree())
    ax.clabel(cs, inline=True, fontsize=6, fmt="%d")
    plt.colorbar(cf, ax=ax, shrink=0.7)

    span = float(gh.max()) - float(gh.min())
    ax.set_title(f"{lev} hPa | ช่วงค่า {span:.0f} gpm", fontsize=10, loc="left")

fig.suptitle(f"geopotential height ทุกระดับ {TIME} UTC", fontsize=13)
plt.tight_layout()
plt.show()

print(f"{'ระดับ':>6} {'ต่ำสุด':>10} {'สูงสุด':>10} {'ช่วงค่า':>10} {'ที่อุบลฯ':>10}")
for lev in levels_hpa:
    gh = ad.geopotential_height(day["z"].sel(level=lev))
    print(f"{lev:>6} {float(gh.min()):>10.1f} {float(gh.max()):>10.1f} "
          f"{float(gh.max()) - float(gh.min()):>10.1f} {ad.value_at(gh, *UBON):>10.1f}")

### ✍️ บันทึกการสังเกต

ช่วงค่า (ตัวเลขคอลัมน์ที่ 4) เปลี่ยนแปลงอย่างไรเมื่อขึ้นไปสูง และบอกอะไรเกี่ยวกับความลึกของระบบ

*(เขียนคำตอบที่นี่)*

## 3. สนามลมสามระดับ — ล่าง กลาง บน

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax, lev in zip(axes, [850, 500, 200]):
    u = day["u"].sel(level=lev)
    v = day["v"].sel(level=lev)
    ws = ad.wind_speed(u, v)

    add_basemap(ax, DOMAIN, gridlines=False)
    cf = ax.contourf(ws["longitude"], ws["latitude"], ws.values,
                     levels=np.arange(0, 31, 2.5), cmap="YlGnBu",
                     extend="max", transform=ccrs.PlateCarree())
    ax.streamplot(u["longitude"].values, u["latitude"].values,
                  u.values, v.values, density=1.3, linewidth=0.6,
                  arrowsize=0.8, color="dimgray",
                  transform=ccrs.PlateCarree())
    plt.colorbar(cf, ax=ax, shrink=0.75, label="m s$^{-1}$")
    ax.set_title(f"{lev} hPa | สูงสุด {float(ws.max()):.1f} m/s",
                 fontsize=11, loc="left")

fig.suptitle(f"สนามลมสามระดับ {TIME} UTC", fontsize=13)
plt.show()

print(f"{'ระดับ':>6} {'สูงสุดในโดเมน':>16} {'ที่อุบลฯ':>12} {'ทิศที่อุบลฯ':>14}")
for lev in [850, 500, 200]:
    u = day["u"].sel(level=lev)
    v = day["v"].sel(level=lev)
    ws = ad.wind_speed(u, v)
    wd = ad.wind_direction(u, v)
    print(f"{lev:>6} {float(ws.max()):>14.1f} m/s {ad.value_at(ws, *UBON):>10.1f} m/s "
          f"{ad.value_at(wd, *UBON):>12.0f}°")

### ✍️ บันทึกการสังเกต

ลมที่อุบลราชธานีแรงหรือเบา และทิศทางเปลี่ยนอย่างไรเมื่อขึ้นไปสูง

*(เขียนคำตอบที่นี่)*

## 4. โปรไฟล์ลมตามแนวดิ่งที่จุดเดียว

In [ ]:
lat_u, lon_u = UBON
levs = ds["level"].values

prof = pd.DataFrame({
    "level": levs,
    "u": [ad.value_at(day["u"].sel(level=l), lat_u, lon_u) for l in levs],
    "v": [ad.value_at(day["v"].sel(level=l), lat_u, lon_u) for l in levs],
})
prof["speed"] = np.sqrt(prof["u"]**2 + prof["v"]**2)
prof["direction"] = (270 - np.degrees(np.arctan2(prof["v"], prof["u"]))) % 360
prof["height_gpm"] = [ad.value_at(ad.geopotential_height(day["z"].sel(level=l)),
                                  lat_u, lon_u) for l in levs]

display(prof.round(2))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 6), sharey=True)

a1.plot(prof["speed"], prof["level"], "o-", color="navy")
a1.set_xlabel("ความเร็วลม (m s$^{-1}$)")
a1.set_ylabel("ระดับความกดอากาศ (hPa)")
a1.invert_yaxis()
a1.set_yscale("log")
a1.set_yticks(levs)
a1.set_yticklabels([str(int(l)) for l in levs])
a1.grid(alpha=0.3)
a1.set_title("ความเร็ว", loc="left")

a2.plot(prof["direction"], prof["level"], "s-", color="darkred")
a2.set_xlabel("ทิศลม (องศา ทิศที่ลมพัดมาจาก)")
a2.set_xlim(0, 360)
a2.set_xticks([0, 90, 180, 270, 360])
a2.set_xticklabels(["N", "E", "S", "W", "N"])
a2.grid(alpha=0.3)
a2.set_title("ทิศทาง", loc="left")

fig.suptitle(f"โปรไฟล์ลมแนวดิ่งที่อุบลราชธานี {TIME} UTC")
plt.tight_layout()
plt.show()

### ✍️ บันทึกการสังเกต

โปรไฟล์ความเร็วมีรูปร่างอย่างไร และทิศลมหมุนไปทางไหนเมื่อขึ้นไปสูง

*(เขียนคำตอบที่นี่)*

## 5. ความหนาของชั้นบรรยากาศ และอุณหภูมิเฉลี่ยของชั้น

In [ ]:
thk = ad.thickness(day["z"], 1000, 500)
tbar = ad.layer_mean_temperature(day["z"], 1000, 500) - 273.15

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                         subplot_kw={"projection": ccrs.PlateCarree()})

add_basemap(axes[0], DOMAIN, gridlines=False)
cf0 = axes[0].contourf(thk["longitude"], thk["latitude"], thk.values,
                       levels=np.arange(5750, 5841, 5), cmap="RdYlBu_r",
                       extend="both", transform=ccrs.PlateCarree())
cs0 = axes[0].contour(thk["longitude"], thk["latitude"], thk.values,
                      levels=np.arange(5750, 5841, 10), colors="black",
                      linewidths=0.6, transform=ccrs.PlateCarree())
axes[0].clabel(cs0, inline=True, fontsize=7, fmt="%d")
plt.colorbar(cf0, ax=axes[0], shrink=0.8, label="gpm")
axes[0].set_title("ความหนาชั้น 1000–500 hPa", fontsize=11, loc="left")

add_basemap(axes[1], DOMAIN, gridlines=False)
cf1 = axes[1].contourf(tbar["longitude"], tbar["latitude"], tbar.values,
                       levels=np.arange(10, 15.1, 0.25), cmap="RdYlBu_r",
                       extend="both", transform=ccrs.PlateCarree())
plt.colorbar(cf1, ax=axes[1], shrink=0.8, label="°C")
axes[1].set_title("อุณหภูมิเฉลี่ยของชั้น 1000–500 hPa", fontsize=11, loc="left")

fig.suptitle(f"โครงสร้างอุณหภูมิของชั้นบรรยากาศ {TIME} UTC", fontsize=13)
plt.show()

low = ad.extreme_location(day["msl"] / 100, "min")
thk_center = ad.value_at(thk, low["lat"], low["lon"])
thk_mean = ad.area_mean(thk)

print(f"ความหนาที่ศูนย์กลางความกดอากาศต่ำ : {thk_center:.1f} gpm")
print(f"ความหนาเฉลี่ยทั้งโดเมน            : {thk_mean:.1f} gpm")
print(f"ผลต่าง                            : {thk_center - thk_mean:+.1f} gpm")
print()
print("warm core (ศูนย์กลางอุ่นกว่ารอบข้าง)" if thk_center > thk_mean
      else "cold core (ศูนย์กลางเย็นกว่ารอบข้าง)")

### ✍️ บันทึกการสังเกต

ศูนย์กลางระบบมีชั้นอากาศหนาหรือบางกว่ารอบข้าง และแปลว่าอะไร

*(เขียนคำตอบที่นี่)*

## 6. ลมเฉือนตามแนวดิ่ง (deep layer shear)

In [ ]:
shear = ad.deep_layer_shear(day["u"], day["v"], lower=850, upper=200)

fig = plt.figure(figsize=(9.5, 7.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, DOMAIN)

cf = ax.contourf(shear["longitude"], shear["latitude"], shear.values,
                 levels=np.arange(0, 41, 2.5), cmap="magma_r",
                 extend="max", transform=ccrs.PlateCarree())
cs = ax.contour(shear["longitude"], shear["latitude"], shear.values,
                levels=[10, 20], colors=["lime", "red"], linewidths=2,
                transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=9, fmt="%d m/s")

low = ad.extreme_location(day["msl"] / 100, "min")
ax.plot(low["lon"], low["lat"], "wx", markersize=14, markeredgewidth=3,
        transform=ccrs.PlateCarree(), zorder=6)

plt.colorbar(cf, ax=ax, shrink=0.8, label="ลมเฉือน 850–200 hPa (m s$^{-1}$)")
ax.set_title(f"ลมเฉือนตามแนวดิ่ง {TIME} UTC\n"
             "เส้นเขียว = 10 m/s · เส้นแดง = 20 m/s · กากบาท = ศูนย์กลางความกดอากาศต่ำ",
             loc="left")
plt.show()

print(f"ลมเฉือนที่ศูนย์กลาง : {ad.value_at(shear, low['lat'], low['lon']):.1f} m/s")
print(f"ลมเฉือนที่อุบลฯ     : {ad.value_at(shear, *UBON):.1f} m/s")
print(f"เฉลี่ยทั้งโดเมน     : {ad.area_mean(shear):.1f} m/s")
print(f"ช่วงค่าในโดเมน      : {float(shear.min()):.1f} ถึง {float(shear.max()):.1f} m/s")

### ✍️ บันทึกการสังเกต

ลมเฉือนที่ศูนย์กลางอยู่ในช่วงไหนของเกณฑ์ และรอบข้างต่างจากศูนย์กลางอย่างไร

*(เขียนคำตอบที่นี่)*

## 7. ความหมุนวนสัมพัทธ์ (relative vorticity)

In [ ]:
u850 = day["u"].sel(level=850)
v850 = day["v"].sel(level=850)

zeta = ad.relative_vorticity(u850, v850) * 1e5     # แปลงเป็นหน่วย 1e-5 s^-1

fig = plt.figure(figsize=(9.5, 7.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, DOMAIN)

cf = ax.contourf(zeta["longitude"], zeta["latitude"], zeta.values,
                 levels=np.arange(-20, 21, 2.5), cmap="RdBu_r",
                 extend="both", transform=ccrs.PlateCarree())

step = 2
ax.barbs(u850["longitude"].values[::step], u850["latitude"].values[::step],
         u850.values[::step, ::step] * 1.94384,
         v850.values[::step, ::step] * 1.94384,
         length=4.5, linewidth=0.4, transform=ccrs.PlateCarree())

peak = ad.extreme_location(zeta, "max")
ax.plot(peak["lon"], peak["lat"], "ko", markersize=10, fillstyle="none",
        markeredgewidth=2, transform=ccrs.PlateCarree(), zorder=6)

plt.colorbar(cf, ax=ax, shrink=0.8,
             label="relative vorticity ที่ 850 hPa ($10^{-5}$ s$^{-1}$)")
ax.set_title(f"ความหมุนวนสัมพัทธ์ {TIME} UTC\n"
             "แดง = หมุนทวนเข็ม (ความกดต่ำ) · น้ำเงิน = หมุนตามเข็ม",
             loc="left")
plt.show()

low = ad.extreme_location(day["msl"] / 100, "min")
print(f"vorticity สูงสุด {peak['value']:.1f} x1e-5 s^-1 "
      f"ที่ {peak['lat']:.1f}N {peak['lon']:.1f}E")
print(f"ความกดอากาศต่ำสุดอยู่ที่ {low['lat']:.1f}N {low['lon']:.1f}E")
print(f"ห่างกัน {abs(peak['lat'] - low['lat']):.1f} องศาละติจูด "
      f"{abs(peak['lon'] - low['lon']):.1f} องศาลองจิจูด")
print(f"\nvorticity ที่อุบลฯ : {ad.value_at(zeta, *UBON):.1f} x1e-5 s^-1")

### ✍️ บันทึกการสังเกต

ตำแหน่ง vorticity สูงสุดตรงกับความกดอากาศต่ำสุดหรือไม่ และบอกอะไรได้

*(เขียนคำตอบที่นี่)*

## 8. การลู่เข้าและแยกตัวของอากาศ (divergence)

In [ ]:
div850 = ad.divergence(day["u"].sel(level=850), day["v"].sel(level=850)) * 1e5
div200 = ad.divergence(day["u"].sel(level=200), day["v"].sel(level=200)) * 1e5

fig, axes = plt.subplots(1, 2, figsize=(15, 6),
                         subplot_kw={"projection": ccrs.PlateCarree()})

low = ad.extreme_location(day["msl"] / 100, "min")

for ax, fld, lev in zip(axes, [div850, div200], [850, 200]):
    add_basemap(ax, DOMAIN, gridlines=False)
    cf = ax.contourf(fld["longitude"], fld["latitude"], fld.values,
                     levels=np.arange(-10, 10.1, 1), cmap="PuOr_r",
                     extend="both", transform=ccrs.PlateCarree())
    ax.plot(low["lon"], low["lat"], "kx", markersize=13, markeredgewidth=3,
            transform=ccrs.PlateCarree(), zorder=6)
    plt.colorbar(cf, ax=ax, shrink=0.8, label="$10^{-5}$ s$^{-1}$")
    ax.set_title(f"divergence ที่ {lev} hPa", fontsize=11, loc="left")

fig.suptitle(f"การลู่เข้า (ม่วง) และแยกตัว (ส้ม) ของอากาศ {TIME} UTC\n"
             "กากบาท = ศูนย์กลางความกดอากาศต่ำ", fontsize=12)
plt.show()

print("ค่าที่ศูนย์กลางความกดอากาศต่ำ")
print(f"  divergence 850 hPa : {ad.value_at(div850, low['lat'], low['lon']):+.1f}")
print(f"  divergence 200 hPa : {ad.value_at(div200, low['lat'], low['lon']):+.1f}")
print()
print("ค่าที่อุบลราชธานี")
print(f"  divergence 850 hPa : {ad.value_at(div850, *UBON):+.1f}")
print(f"  divergence 200 hPa : {ad.value_at(div200, *UBON):+.1f}")
print()
print("ช่วงค่าทั้งโดเมน")
print(f"  850 hPa : {float(div850.min()):+.1f} ถึง {float(div850.max()):+.1f}")
print(f"  200 hPa : {float(div200.min()):+.1f} ถึง {float(div200.max()):+.1f}")

### ✍️ บันทึกการสังเกต

เปรียบเทียบเครื่องหมายของ divergence ที่ 850 กับ 200 hPa ตรงศูนย์กลาง ตรงกับที่คาดไว้หรือไม่ ถ้าไม่ตรง คิดว่าเพราะอะไร

*(เขียนคำตอบที่นี่)*

## 9. การพาความร้อนในแนวราบ (temperature advection)

In [ ]:
tbar_K = ad.layer_mean_temperature(day["z"], 1000, 500)
adv = ad.temperature_advection(day["u"].sel(level=850),
                               day["v"].sel(level=850),
                               tbar_K) * 86400          # K/s -> K/day

fig = plt.figure(figsize=(9.5, 7.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, DOMAIN)

cf = ax.contourf(adv["longitude"], adv["latitude"], adv.values,
                 levels=np.arange(-10, 10.1, 1), cmap="coolwarm",
                 extend="both", transform=ccrs.PlateCarree())

cs = ax.contour(tbar_K["longitude"], tbar_K["latitude"],
                (tbar_K - 273.15).values, levels=np.arange(10, 15, 0.5),
                colors="black", linewidths=0.5, transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=7, fmt="%.1f")

plt.colorbar(cf, ax=ax, shrink=0.8, label="temperature advection (K/day)")
ax.set_title(f"การพาความร้อนที่ 850 hPa {TIME} UTC\n"
             "แดง = พาอากาศอุ่นเข้ามา · น้ำเงิน = พาอากาศเย็นเข้ามา\n"
             "เส้นดำ = อุณหภูมิเฉลี่ยชั้น 1000–500 hPa (°C)", loc="left")
plt.show()

print(f"advection ที่อุบลฯ : {ad.value_at(adv, *UBON):+.2f} K/day")
low = ad.extreme_location(day["msl"] / 100, "min")
print(f"advection ที่ศูนย์กลาง : {ad.value_at(adv, low['lat'], low['lon']):+.2f} K/day")
print(f"ช่วงค่าในโดเมน : {float(adv.min()):+.2f} ถึง {float(adv.max()):+.2f} K/day")
print(f"ค่าเฉลี่ยทั้งโดเมน : {ad.area_mean(adv):+.3f} K/day")

### ✍️ บันทึกการสังเกต

ที่อุบลราชธานีมีการพาอากาศอุ่นหรือเย็นเข้ามา และค่ามากพอจะอธิบายอุณหภูมิที่เห็นไหม

*(เขียนคำตอบที่นี่)*

## 10. ลม geostrophic เทียบกับลมจริง

In [ ]:
z500 = day["z"].sel(level=500)
ug, vg = ad.geostrophic_wind(z500)
u_real = day["u"].sel(level=500)
v_real = day["v"].sel(level=500)

ws_g = ad.wind_speed(ug, vg)
ws_r = ad.wind_speed(u_real, v_real)
u_ag = u_real - ug
v_ag = v_real - vg
ws_ag = ad.wind_speed(u_ag, v_ag)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5),
                         subplot_kw={"projection": ccrs.PlateCarree()})

titles = ["ลมจริง", "ลม geostrophic (ตามทฤษฎี)", "ลม ageostrophic (ส่วนต่าง)"]
fields = [ws_r, ws_g, ws_ag]
winds = [(u_real, v_real), (ug, vg), (u_ag, v_ag)]

for ax, fld, (uu, vv), ttl in zip(axes, fields, winds, titles):
    add_basemap(ax, DOMAIN, gridlines=False)
    cf = ax.contourf(fld["longitude"], fld["latitude"], fld.values,
                     levels=np.arange(0, 31, 2.5), cmap="YlOrBr",
                     extend="max", transform=ccrs.PlateCarree())
    s = 3
    ax.quiver(uu["longitude"].values[::s], uu["latitude"].values[::s],
              uu.values[::s, ::s], vv.values[::s, ::s],
              scale=350, width=0.004, transform=ccrs.PlateCarree())
    plt.colorbar(cf, ax=ax, shrink=0.75, label="m s$^{-1}$")
    ax.set_title(ttl, fontsize=11, loc="left")

fig.suptitle(f"ลมจริง ลมตามทฤษฎี และส่วนต่าง ที่ 500 hPa {TIME} UTC", fontsize=13)
plt.show()

print(f"ความเร็วลมจริงสูงสุด        : {float(ws_r.max()):.1f} m/s")
print(f"ความเร็วลม geostrophic สูงสุด: {float(ws_g.max()):.1f} m/s")
print(f"ลม ageostrophic สูงสุด      : {float(ws_ag.max()):.1f} m/s")
print(f"สัดส่วน ageostrophic ต่อลมจริง: {float(ws_ag.max()) / float(ws_r.max()) * 100:.0f}%")

### ✍️ บันทึกการสังเกต

ลมจริงกับลมตามทฤษฎีต่างกันมากไหม และบอกอะไรเกี่ยวกับการใช้ทฤษฎีนี้ในเขตร้อน

*(เขียนคำตอบที่นี่)*

## 11. แผนภาพ Hovmöller — ฝนเปลี่ยนแปลงตามเวลาและลองจิจูด

In [ ]:
band = ds["tp"].sel(latitude=slice(13, 19))     # แถบละติจูดภาคอีสาน
hov = band.mean(dim="latitude")

fig, ax = plt.subplots(figsize=(11, 7))

mesh = ax.pcolormesh(hov["longitude"], hov["time"], hov.values,
                     cmap="GnBu", shading="auto", vmin=0, vmax=12)
plt.colorbar(mesh, ax=ax, label="ฝนเฉลี่ยในแถบ 13–19°N (mm / 6 ชม.)")

ax.axhline(pd.Timestamp("2021-09-24"), color="crimson",
           linestyle="--", linewidth=2)
ax.text(ds["longitude"].min() + 0.5, pd.Timestamp("2021-09-24"),
        " 24 ก.ย. (D0)", color="crimson", va="bottom", fontweight="bold")

ax.set_xlabel("ลองจิจูด (°E)")
ax.set_ylabel("เวลา (UTC)")
ax.set_title("Hovmöller — ฝนเฉลี่ยตามลองจิจูดและเวลา\n"
             "แถบเอียงลงซ้าย = ระบบเคลื่อนไปทางตะวันตก", loc="left")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

peak_time = hov.max(dim="longitude").idxmax().values
peak_lon = hov.max(dim="time").idxmax().values
print(f"ช่วงเวลาที่ฝนเฉลี่ยในแถบสูงสุด : {str(peak_time)[:16]} UTC")
print(f"ลองจิจูดที่ฝนสะสมมากที่สุด     : {float(peak_lon):.1f}°E")
print(f"ค่าสูงสุดในแผนภาพ              : {float(hov.max()):.1f} mm / 6 ชม.")

### ✍️ บันทึกการสังเกต

แถบฝนเอียงไปทางไหน และมีช่วงไหนที่แถบตั้งตรง (ระบบหยุดนิ่ง) บ้าง

*(เขียนคำตอบที่นี่)*

## 12. อนุกรมเวลาที่จุดเดียว ตลอดทั้งวัน

In [ ]:
lat_u, lon_u = UBON
day_sel = ds.sel(time="2021-09-24")

rows = []
for t in day_sel["time"].values:
    s = ds.sel(time=t)
    u8 = ad.value_at(s["u"].sel(level=850), lat_u, lon_u)
    v8 = ad.value_at(s["v"].sel(level=850), lat_u, lon_u)
    rows.append({
        "เวลา UTC": str(t)[11:16],
        "เวลาไทย": (pd.Timestamp(t) + pd.Timedelta(hours=7)).strftime("%H:%M"),
        "MSLP (hPa)": round(ad.value_at(s["msl"], lat_u, lon_u) / 100, 1),
        "T2m (°C)": round(ad.value_at(s["t2m"], lat_u, lon_u) - 273.15, 1),
        "ฝน (mm/6ชม.)": round(ad.value_at(s["tp"], lat_u, lon_u), 1),
        "ลม850 (m/s)": round(float(np.sqrt(u8**2 + v8**2)), 1),
        "ทิศ850": round(float((270 - np.degrees(np.arctan2(v8, u8))) % 360)),
    })

tbl = pd.DataFrame(rows)
display(tbl)

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
x = range(len(tbl))

axes[0, 0].plot(x, tbl["MSLP (hPa)"], "o-", color="navy")
axes[0, 0].set_ylabel("ความกดอากาศ (hPa)")
axes[0, 0].invert_yaxis()

axes[0, 1].bar(x, tbl["ฝน (mm/6ชม.)"], color="teal", alpha=0.7)
axes[0, 1].set_ylabel("ฝน (mm / 6 ชม.)")

axes[1, 0].plot(x, tbl["T2m (°C)"], "s-", color="darkred")
axes[1, 0].set_ylabel("อุณหภูมิ (°C)")

axes[1, 1].plot(x, tbl["ลม850 (m/s)"], "^-", color="darkgreen")
axes[1, 1].set_ylabel("ความเร็วลม 850 hPa (m/s)")

for ax in axes.flat:
    ax.set_xticks(list(x))
    ax.set_xticklabels(tbl["เวลาไทย"])
    ax.grid(alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("เวลาไทย")

fig.suptitle("อุบลราชธานี 24 กันยายน 2564 — การเปลี่ยนแปลงภายในวัน")
plt.tight_layout()
plt.show()

print(f"ฝนรวมทั้งวันที่จุดนี้ : {tbl['ฝน (mm/6ชม.)'].sum():.1f} mm")

### ✍️ บันทึกการสังเกต

ความกดอากาศต่ำสุดเวลาไหน และลมแรงขึ้นหรืออ่อนลงเมื่อระบบเข้าใกล้

*(เขียนคำตอบที่นี่)*

## 13. สรุปการวินิจฉัย

In [ ]:
low = ad.extreme_location(day["msl"] / 100, "min")
zeta = ad.relative_vorticity(day["u"].sel(level=850), day["v"].sel(level=850))
div850 = ad.divergence(day["u"].sel(level=850), day["v"].sel(level=850))
div200 = ad.divergence(day["u"].sel(level=200), day["v"].sel(level=200))
shear = ad.deep_layer_shear(day["u"], day["v"])
thk = ad.thickness(day["z"])
ws850 = ad.wind_speed(day["u"].sel(level=850), day["v"].sel(level=850))

summary = pd.DataFrame([
    ["ความกดอากาศต่ำสุด", f"{low['value']:.1f} hPa",
     f"{low['lat']:.1f}N {low['lon']:.1f}E"],
    ["vorticity สูงสุด 850 hPa", f"{float(zeta.max())*1e5:.1f} x1e-5 s^-1", "-"],
    ["divergence ที่ศูนย์กลาง 850", f"{ad.value_at(div850, low['lat'], low['lon'])*1e5:+.1f} x1e-5 s^-1", "-"],
    ["divergence ที่ศูนย์กลาง 200", f"{ad.value_at(div200, low['lat'], low['lon'])*1e5:+.1f} x1e-5 s^-1", "-"],
    ["ลมเฉือน 850-200 ที่ศูนย์กลาง", f"{ad.value_at(shear, low['lat'], low['lon']):.1f} m/s", "-"],
    ["ความหนา 1000-500 ที่ศูนย์กลาง", f"{ad.value_at(thk, low['lat'], low['lon']):.1f} gpm",
     f"เฉลี่ยโดเมน {ad.area_mean(thk):.1f}"],
    ["ลม 850 สูงสุดในโดเมน", f"{float(ws850.max()):.1f} m/s", "-"],
    ["ฝน 6 ชม. สูงสุดในโดเมน", f"{float(day['tp'].max()):.1f} mm", "-"],
    ["ฝนที่อุบลฯ รวมทั้งวัน", f"{float(ds['tp'].sel(time='2021-09-24').sel(latitude=UBON[0], longitude=UBON[1], method='nearest').sum()):.1f} mm", "-"],
], columns=["ตัวแปรวินิจฉัย", "ค่า", "หมายเหตุ"])

display(summary)

### ✍️ บันทึกการสังเกต

เขียนสรุปด้วยคำของตัวเอง 5–10 บรรทัด ว่าวันนี้บรรยากาศเป็นอย่างไร และอะไรทำให้ฝนตก

*(เขียนคำตอบที่นี่)*

---

## แบบฝึกหัดท้ายบท

1. เปลี่ยน `TIME` เป็น `"2021-09-23T12:00"` (หนึ่งวันก่อนเกิดเหตุ) แล้วรันทั้งบทใหม่
   เปรียบเทียบว่าตัวแปรวินิจฉัยตัวไหนเปลี่ยนมากที่สุด และตัวไหนเปลี่ยนน้อยที่สุด
2. เปลี่ยน `UBON` เป็นพิกัดบ้านเกิดของตัวเอง แล้วทำหัวข้อ 4 และ 12 ใหม่
   อธิบายว่าสภาพอากาศที่บ้านวันนั้นต่างจากอุบลราชธานีอย่างไร
3. คำนวณ **absolute vorticity** (ใช้ `ad.absolute_vorticity`) แล้วเทียบกับ
   relative vorticity ว่าต่างกันอย่างไร และอธิบายว่าส่วนต่างมาจากไหน
4. สร้างแผนที่ divergence ที่ระดับ 700 hPa เพิ่มเข้าไปในหัวข้อ 8
   แล้วอธิบายว่าชั้นกลางสอดคล้องกับชั้นล่างและชั้นบนหรือไม่
5. เขียนรายงานสั้น 1 หน้า สรุปสภาพบรรยากาศของวันนั้น โดยอ้างตัวเลขจากอย่างน้อย
   5 หัวข้อในบทนี้ และแนบแผนที่อย่างน้อย 3 รูป